In [1]:
import pickle
import numpy as np
import torch
import torch.nn.functional as F
from torch.nn import Linear
from torch_geometric.nn import SAGEConv
from torch_geometric.data import Data
from torch_geometric.loader import NeighborLoader
import psutil

In [2]:
print(f"RAM: {psutil.virtual_memory().percent:.1f}%")

RAM: 11.9%


In [6]:
# upload graph and verify intact

with open('../data/canwell_graph.pkl', 'rb') as f:
    data = pickle.load(f)

G = data['G']
print(f"Nodes: {G.number_of_nodes():,}")
print(f"Edges: {G.number_of_edges():,}")

# check a random node
import random
node = random.choice(list(G.nodes()))
print(G.nodes[node])

Nodes: 4,792,230
Edges: 19,151,335
{'dem_idx': (493, 883), 'elev': 1092.7264404296875, 'slope': 27.461088180541992, 'aspect': 229.01515197753906, 'doubslope': 67.7156753540039, 'diff': -0.7021891474723816, 'is_slope': True, 'is_basin': False}


In [ ]:
# convert networkx graph to a pytorch data object

print("building node feature matrix...")

num_features = 4
nodes = list(G.nodes(data=True))
node_list = [n for n, _ in nodes]
node_to_idx = {n: i for i, n in enumerate(node_list)}

# build feature matrix: [elev, slope, aspect, doubslope]
# diff handled as the target
num_nodes = len(node_list)
x = torch.zeros((num_nodes, num_features), dtype=torch.float)
y = torch.zeros(num_nodes, dtype=torch.float)
is_slope = torch.zeros(num_nodes, dtype=torch.bool)
is_basin = torch.zeros(num_nodes, dtype=torch.bool)
has_diff = torch.zeros(num_nodes, dtype=torch.bool)

for i, (n,attr) in enumerate(nodes):
    x[i, 0] = attr['elev'] if attr['elev'] is not None else 0.0
    x[i, 1] = attr['slope'] if attr['slope'] is not None else 0.0
    x[i, 2] = attr['aspect'] if attr['aspect'] is not None else 0.0
    x[i, 3] = attr['doubslope'] if attr['doubslope'] is not None else 0.0

    if attr['diff'] is not None:
        y[i] = attr['diff']
        has_diff[i] = True

    is_slope[i] = attr['is_slope']
    is_basin[i] = attr['is_basin']

print(f"feature matrix shape: {x.shape}")
print(f"nodes with diff values: {has_diff.sum():,}")
print(f"slope nodes: {is_slope.sum():,}")
print(f"basin nodes: {is_basin.sum():,}")
print(f"RAM: {psutil.virtual_memory().percent:.1f}%")

In [12]:
# How many nodes had None values that got filled with 0
none_slope    = sum(1 for _, attr in nodes if attr['slope']     is None)
none_aspect   = sum(1 for _, attr in nodes if attr['aspect']    is None)
none_doubslop = sum(1 for _, attr in nodes if attr['doubslope'] is None)
none_elev     = sum(1 for _, attr in nodes if attr['elev']      is None)

print(f"None values filled with 0:")
print(f"  elev:      {none_elev:,}")
print(f"  slope:     {none_slope:,}")
print(f"  aspect:    {none_aspect:,}")
print(f"  doubslope: {none_doubslop:,}")
print(f"  diff missing: {(~has_diff).sum():,}")
print(f"RAM: {psutil.virtual_memory().percent:.1f}%")

None values filled with 0:
  elev:      0
  slope:     21
  aspect:    21
  doubslope: 35
  diff missing: 11,977
RAM: 93.2%


In [18]:
torch.save({
    'x': x,
    'y': y,
    'is_slope': is_slope,
    'is_basin': is_basin,
    'has_diff': has_diff,
    'node_to_idx': node_to_idx
}, 'canwell_features.pt')

print("saved")

saved


In [3]:
# after this, consider restarting the kernel
# run the imports and ram check from cells 1 and 2

In [4]:
print("loading features...")
feat = torch.load('canwell_features.pt')
node_to_idx = feat['node_to_idx']
print(f"ram after features load: {psutil.virtual_memory().percent:.1f}%")

loading features...
ram after features load: 15.6%


In [6]:
print("loading graph...")
with open('../data/canwell_graph.pkl', 'rb') as f:
    data = pickle.load(f)
G = data['G']
print(f"ram after G load: {psutil.virtual_memory().percent:.1f}%")

loading graph...
ram after G load: 89.4%


In [7]:
del data
import gc
gc.collect()
print(f"ram after del data: {psutil.virtual_memory().percent:.1f}%")

ram after del data: 89.3%


In [8]:
print("building edge index...")
n = G.number_of_edges()
src = np.empty(n, dtype=np.int32)
dst = np.empty(n, dtype=np.int32)

for i, (u,v) in enumerate(G.edges()):
    if i%2000000 == 0:
        print(f" {i:,}/{n:,}, ram: {psutil.virtual_memory().percent:.1f}%")
    src[i] = node_to_idx[u]
    dst[i] = node_to_idx[v]

del G
import gc
gc.collect()
print(f"ram after del G: {psutil.virtual_memory().percent:.1f}%")

building edge index...
 0/19,151,335, ram: 89.4%
 2,000,000/19,151,335, ram: 89.5%
 4,000,000/19,151,335, ram: 89.9%
 6,000,000/19,151,335, ram: 90.2%
 8,000,000/19,151,335, ram: 90.3%
 10,000,000/19,151,335, ram: 90.4%
 12,000,000/19,151,335, ram: 91.0%
 14,000,000/19,151,335, ram: 91.1%
 16,000,000/19,151,335, ram: 91.2%
 18,000,000/19,151,335, ram: 91.3%
ram after del G: 17.0%


In [11]:
# build undirected edge index
edge_index = torch.tensor(
    np.vstack([np.concatenate([src,dst]),
               np.concatenate([dst,src])]),
    dtype=torch.long
)

del src, dst
gc.collect()

print(f"edge index shape: {edge_index.shape}")
print(f"ram: {psutil.virtual_memory().percent:.1f}%")

edge index shape: torch.Size([2, 38302670])
ram: 19.9%


In [12]:
torch.save(edge_index, 'canwell_edgidx.pt')
print("saved")

saved
